# ممیزی پوشش دادهٔ Divar/Laya — مرحله‌های v3 و v6

این notebook فقط manifestها و خروجی aggregate ممیزی پوشش را می‌خواند؛ هیچ متن آگهی، شناسه، فایل corpus، checkpoint یا مدل را تغییر نمی‌دهد.

## tl;dr

- از ۱٬۰۰۰٬۰۰۰ ردیف منبع، ۹۹۹٬۴۱۶ آگهی نرمال شده؛ ۵۸۴ مورد به‌علت تعارض (۵۸۳) یا دستهٔ نگاشت‌نشده (۱) کنار گذاشته شده‌اند.
- پوشش منبع فقط ۱۶ از ۱۸ دستهٔ ملکی محصول است؛ `agency-services` و `land-rent` در منبع فعلی نمونه ندارند.
- ۳۹۷ از ۱٬۲۰۷ کاتالوگ شهر برنامه در منبع نگاشت شده‌اند (۳۲٫۸۹٪)، با وجود اینکه ۹۶٫۷۷٪ آگهی‌های نرمال‌شده شهر نگاشت‌شده دارند. پوشش سطری محله فقط ۴۳٫۰۵٪ است.
- v3 و v6 هر دو synthetic و بدون ground truth یا بازبینی انسانی‌اند. در هر دو manifest، budget/monthly_rent/deposit و usage صددرصد `unknown` هستند؛ v6 آموزش را فقط برای سه تصمیم category/property/transaction تعریف می‌کند.

نتیجه: این corpus برای آزمایش تحقیقاتی محدود قابل حسابرسی است، اما نه دیتاست واقعیِ نیاز کاربران، نه پوشش کامل شهر/دسته و نه مدرک دقت محصول.

## Context & Methods

**واحد تحلیل:** یک ردیفِ نیاز فرضیِ نرمال‌شده به‌همراه targetهای typed؛ ردیف‌ها از آگهی عرضهٔ Divar مشتق شده‌اند و مشاهدهٔ رفتار متقاضی نیستند.

**ورودی‌ها:** manifest فکت‌های Divar نسخهٔ v4، crosswalk محله، manifest دیتاست‌های hypothetical v3/v6، و audit read-only کاتالوگ واقعی دسته/شهر در برنامه.

**روش:** اجرای `bun scripts/datasets/audit-divar-app-coverage.ts` برای نسبت دسته/شهر/محله؛ سپس جمع unknown-rate از `decisionCounts` در manifestهای v3 و v6. محاسبه‌ها aggregate-only هستند؛ ردیف خامی چاپ یا ارسال نمی‌شود.

### Key Assumptions

- شمارش شهرهای کاتالوگ با شمارش شهرهای دارای منبع یکی نیست؛ نسبت ۳۲٫۸۹٪ پوشش کاتالوگ است، و ۹۶٫۷۷٪ پوشش ردیف‌های دارای شهر نگاشت‌شده.
- پوشش جفت محلهٔ source (۹۶٫۹۷٪) با پوشش ردیفی corpus (۴۳٫۰۵٪) متفاوت است؛ اولی فقط نام‌های مشاهده‌شده در ۱٬۱۸۸ جفت شهر-محله را می‌سنجد.
- Unknown در فیلدهای عددی به معنی دادهٔ واقعیِ نیاز نیست؛ این فیلدها خارج از سه target اصلی fine-tune v6 هستند و باید در مسیر parser/فرم جداگانه ارزیابی شوند.

## Data

فایل‌های محلی دقیق: `data/divar/divar-property-offer-facts-v4-exact-current-city-map-2026-09-28.jsonl.manifest.json`، `data/divar/official-neighborhood-app-crosswalk-2026-09-28-legacy-aliases.json`، `data/divar/divar-hypothetical-needs-laya-v3-exact-city-aliases-2026-09-28.jsonl.manifest.json` و `data/divar/divar-hypothetical-needs-v6-explicit-rent-mode-2026-09-28.jsonl.manifest.json`.

سلول بعدی خلاصه را از همین artifactها دوباره محاسبه می‌کند؛ آن را از ریشهٔ مخزن و با Bun اجرا کنید.

In [1]:
import json
import subprocess
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'data/divar').is_dir():
    raise RuntimeError('Run this notebook from the repository root.')

def read_manifest(name):
    path = ROOT / 'data/divar' / f'{name}.jsonl.manifest.json'
    return json.loads(path.read_text(encoding='utf-8'))

coverage = json.loads(subprocess.run(
    ['bun', 'scripts/datasets/audit-divar-app-coverage.ts'],
    cwd=ROOT, capture_output=True, text=True, check=True,
).stdout)
v3 = read_manifest('divar-hypothetical-needs-laya-v3-exact-city-aliases-2026-09-28')
v6 = read_manifest('divar-hypothetical-needs-v6-explicit-rent-mode-2026-09-28')

def unknown_pct(manifest, fields):
    result = {}
    for field in fields:
        counts = manifest['decisionCounts'][field]
        total = sum(counts.values())
        result[field] = round(100 * counts.get('unknown', 0) / total, 2)
    return result

numeric_and_context_fields = ['transaction_type', 'city', 'neighborhood', 'usage', 'budget', 'monthly_rent', 'deposit']
summary = {
    'source': {
        'rowsRead': coverage['sourceRowsRead'],
        'normalizedOffers': coverage['normalizedOfferRows'],
        'skipped': coverage['sourceRowReconciliation']['rowsSkippedByNormalizer'],
        'skippedBreakdown': coverage['sourceRowReconciliation']['skippedBreakdown'],
    },
    'categories': {
        'app': coverage['appCategories']['count'],
        'source': coverage['appCategories']['sourceCategoryCount'],
        'missingFromSource': coverage['appCategories']['missingFromSource'],
        'coveragePct': round(100 * coverage['appCategories']['sourceCategoryCount'] / coverage['appCategories']['count'], 2),
    },
    'cities': {
        'appCatalogs': coverage['appLocations']['cityCatalogs'],
        'sourceCities': coverage['sourceCityCoverage']['sourceCities'],
        'mappedSourceCities': coverage['sourceCityCoverage']['sourceCitiesMappedToApp'],
        'shareAppCatalogCoveredPct': coverage['sourceCityCoverage']['shareOfAppCityCatalogsCoveredPct'],
        'normalizedRowsWithCityPct': coverage['sourceCityCoverage']['shareOfNormalizedOffersWithMappedCityPct'],
    },
    'neighborhoods': {
        'appNeighborhoodIds': coverage['appLocations']['uniqueCityNeighborhoodIds'],
        'observedPairs': coverage['sourceNeighborhoodCoverage']['sourceCityNeighborhoodPairs'],
        'resolvedPairsIncludingAliases': coverage['sourceNeighborhoodCoverage']['resolvedPairsIncludingLegacyAliases'],
        'pairCoveragePct': coverage['sourceNeighborhoodCoverage']['pairCoverageIncludingLegacyAliasesPct'],
        'normalizedRowsWithMappedNeighborhoodPct': coverage['sourceNeighborhoodCoverage']['shareOfNormalizedOffersWithMappedNeighborhoodPct'],
    },
    'training': {
        'v3Rows': v3['outputRows'],
        'v6Rows': v6['outputRows'],
        'v3UnknownPct': unknown_pct(v3, numeric_and_context_fields),
        'v6UnknownPct': unknown_pct(v6, numeric_and_context_fields),
        'realNeedGroundTruth': v3['realNeedGroundTruth'] and v6['realNeedGroundTruth'],
        'humanReviewed': v3['humanReviewed'] and v6['humanReviewed'],
    },
}
assert summary['source']['rowsRead'] == 1_000_000
assert summary['categories']['missingFromSource'] == ['agency-services', 'land-rent']
assert summary['training']['v3UnknownPct']['budget'] == 100.0
assert summary['training']['v6UnknownPct']['monthly_rent'] == 100.0
assert summary['training']['realNeedGroundTruth'] is False
print(json.dumps(summary, ensure_ascii=False, indent=2))

{
  "source": {
    "rowsRead": 1000000,
    "normalizedOffers": 999416,
    "skipped": 584,
    "skippedBreakdown": {
      "conflict": 583,
      "missingText": 0,
      "unmappedCategory": 1
    }
  },
  "categories": {
    "app": 18,
    "source": 16,
    "missingFromSource": [
      "agency-services",
      "land-rent"
    ],
    "coveragePct": 88.89
  },
  "cities": {
    "appCatalogs": 1207,
    "sourceCities": 421,
    "mappedSourceCities": 397,
    "shareAppCatalogCoveredPct": 32.89,
    "normalizedRowsWithCityPct": 96.77
  },
  "neighborhoods": {
    "appNeighborhoodIds": 48110,
    "observedPairs": 1188,
    "resolvedPairsIncludingAliases": 1152,
    "pairCoveragePct": 96.97,
    "normalizedRowsWithMappedNeighborhoodPct": 43.05
  },
  "training": {
    "v3Rows": 723080,
    "v6Rows": 738109,
    "v3UnknownPct": {
      "transaction_type": 0.4,
      "city": 1.59,
      "neighborhood": 51.33,
      "usage": 100.0,
      "budget": 100.0,
      "monthly_rent": 100.0,
      "dep

## Results

| موضوع | نتیجه | پیامد برای آموزش |
|---|---:|---|
| دسته‌های موجود | ۱۶ از ۱۸ (۸۸٫۸۹٪) | دو دسته هیچ مثال Divar ندارند؛ با منبع مجاز فعلی نمی‌توان ادعای پوشش کامل کرد. |
| شهرهای کاتالوگ دارای منبع نگاشت‌شده | ۳۹۷ از ۱٬۲۰۷ (۳۲٫۸۹٪) | پوشش سطری بالا، توزیع را برای شهرهای کم‌داده تضمین نمی‌کند. |
| آگهی‌های دارای شهر نگاشت‌شده | ۹۶٫۷۷٪ | ۱٫۵۹٪ تا ۱٫۶۸٪ ردیف‌های v3/v6 شهر unknown دارند. |
| آگهی‌های دارای محلهٔ نگاشت‌شده | ۴۳٫۰۵٪ | زوج‌های مشاهده‌شده خوب resolve می‌شوند، اما بیشتر ردیف‌ها label محله ندارند. |
| ردیف‌های v3 / v6 | ۷۲۳٬۰۸۰ / ۷۳۸٬۱۰۹ | هر دو از عرضه به نیاز فرضی تبدیل شده‌اند؛ حجم به معنی تنوع یا gold label نیست. |
| budget / monthly_rent / deposit | ۱۰۰٪ unknown در هر دو | از این manifestها نمی‌توان توانایی استخراج بودجه/اجاره را نتیجه گرفت. |

**شدت:** نبود ground truth واقعی و human review برای کاربرد محصول بحرانی است؛ برای آزمایش الگوریتمی صرفاً یک محدودیت شناخته‌شده است. پوشش دسته و شهر برای ادعای «تمام شهرها و دسته‌ها» زیاد و با اطمینان بالا ناقص است.

## Takeaways

1. مرحلهٔ v3 اکنون زنده است و به checkpoint دست نمی‌زنیم؛ v6 هم فقط category/property/transaction را fine-tune می‌کند.
2. هیچ ردیف فرضی را به ground truth یا پوشش شهر/دسته ارتقا ندهیم. v6 را صرفاً آزمایش پژوهشی نگه داریم و نتیجه‌اش را روی نیاز واقعی ارزیابی نکنیم.
3. برای رسیدن به هدف پوشش واقعی با محدودیت «فقط Divar»، ابتدا باید دسته‌های گمشده و شهر/محله‌های کم‌پوشش را از همان منبع به‌صورت aggregate ممیزی کرد؛ اگر منبع نمونه ندارد، شکاف باید آشکار بماند.
4. توانایی عددی budget/rent/deposit باید جداگانه روی corpus واقعیِ درخواست‌ها یا benchmark دستیِ دارای مجوز سنجیده شود؛ این دو manifest آن را آموزش یا اثبات نمی‌کنند.
5. حقوق ODbL، privacy، نام‌ها/اطلاعات غیرمستقیم و مجوز استفاده برای آموزش هنوز گیت‌اند؛ هیچ خروجی برای انتشار یا بارگذاری مدل production مجاز نیست.

**محدودیت اجرا:** محیط پروژه `nbformat`/`nbclient` و Jupyter را نصب ندارد. کد سلول با Python استاندارد از روی متن notebook اجرا و خروجی ثبت‌شده بازبینی شده؛ برای اجرای تعاملی کامل، نصب این ابزارها لازم است.